# 7. MODEL DEPLOYMENT PREPARATION
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../models/churn_model_YYYYMMDD.joblib`, `../data/processed/churn_features_YYYYMMDD.parquet`, and `../data/processed/churn_explainability_YYYYMMDD.parquet`

**OUTPUT:** `../models/churn_scoring_package_YYYYMMDD.joblib`, `../src/models/churn_scoring.py`, and `../data/processed/churn_inference_smoke_test_YYYYMMDD.parquet`

*A reusable scoring package with deployment-ready metadata, inference helpers, and a smoke-tested prediction example.*


---
## 7.1. STARTING SITUATION


The canonical V2C model is now trained, diagnosed, and explained. Before orchestration can consume it, the project needs a stable inference package that reproduces the same feature alignment, percentile-based risk-tier logic, and core retention-rule outputs every time the model runs.

---
## 7.2. NOTEBOOK OBJECTIVE


- **Business objective:** package the churn model so daily scoring can run reliably and produce outputs that downstream systems can consume without manual intervention.
- **Analytical objective:** create a deployment-ready artifact, a plain Python scoring module, and a smoke test proving end-to-end inference works on recent customer snapshots under the canonical V2C policy.

In [1]:
import json
import logging
from textwrap import dedent
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb07_deployment_prep')
logger.info('NB07 started: model deployment preparation.')


2026-05-21 19:19:44,017 | INFO | NB07 started: model deployment preparation.


In [2]:
def resolve_project_root() -> Path:
    candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidate_roots:
        if (candidate / 'data' / 'processed').exists() and (candidate / 'models').exists():
            return candidate
        nested_candidate = candidate / 'daily-customer-churn-predictor'
        if (nested_candidate / 'data' / 'processed').exists() and (nested_candidate / 'models').exists():
            return nested_candidate
    raise FileNotFoundError('Could not resolve project root containing data/processed and models directories.')


def extract_artifact_tag(path: Path) -> str:
    return path.stem.split('_')[-1]


def resolve_required_artifact(directory: Path, prefix: str, run_date_tag: str, suffix: str) -> Path:
    candidate = directory / f'{prefix}_{run_date_tag}.{suffix}'
    if not candidate.exists():
        raise FileNotFoundError(f'Missing required artifact for run_date_tag={run_date_tag}: {candidate}')
    return candidate


PROJECT_ROOT = resolve_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_MODELS_DIR = PROJECT_ROOT / 'src' / 'models'
SRC_MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_path = sorted(MODELS_DIR.glob('churn_model_*.joblib'))[-1]
package = joblib.load(model_path)
run_date_tag = package.get('run_date_tag', extract_artifact_tag(model_path))
run_id = package.get('run_id', f'canonical_v2c_{run_date_tag}')
feature_path = resolve_required_artifact(PROCESSED_DIR, 'churn_features', run_date_tag, 'parquet')
explainability_path = resolve_required_artifact(PROCESSED_DIR, 'churn_explainability', run_date_tag, 'parquet')
prediction_path = resolve_required_artifact(PROCESSED_DIR, 'churn_predictions', run_date_tag, 'parquet')
scoring_package_path = MODELS_DIR / f'churn_scoring_package_{run_date_tag}.joblib'
smoke_test_path = PROCESSED_DIR / f'churn_inference_smoke_test_{run_date_tag}.parquet'
scoring_module_path = SRC_MODELS_DIR / 'churn_scoring.py'

logger.info('Deployment package anchored to run_date_tag=%s', run_date_tag)
logger.info('Deployment package anchored to run_id=%s', run_id)


2026-05-21 19:19:44,698 | INFO | Deployment package anchored to run_date_tag=20260506


2026-05-21 19:19:44,699 | INFO | Deployment package anchored to run_id=canonical_v2c_20260506


In [3]:
feature_df = pd.read_parquet(feature_path)
explainability_df = pd.read_parquet(explainability_path)
prediction_df = pd.read_parquet(prediction_path)
model = package['model']
feature_columns = package['feature_columns']
target_column = package.get('target_column', 'churn_v2_label' if 'churn_v2_label' in feature_df.columns else 'churn_90d_label')
version_name = package.get('version_name', 'v2')
model_version = package.get('model_version', f"{version_name}_{run_date_tag}")
pipeline_tag = package.get('pipeline_tag', 'canonical_v2c_phase2')

risk_thresholds = {
    'medium_min_score': float(prediction_df['churn_probability'].quantile(0.50)),
    'high_min_score': float(prediction_df['churn_probability'].quantile(0.80)),
    'quantile_policy': {'low': '0%-50%', 'medium': '50%-80%', 'high': '80%-100%'}
}

driver_policy = (
    explainability_df.groupby(['risk_tier', 'top_driver_group', 'recommended_offer_type'], observed=False)
    .size()
    .reset_index(name='rows_n')
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)

dominant_driver_policy = (
    driver_policy.groupby('risk_tier', observed=False)
    .first()
    .reset_index()[['risk_tier', 'top_driver_group', 'recommended_offer_type']]
)

inference_metadata = {
    'model_name': package['model_name'],
    'version_name': version_name,
    'model_version': model_version,
    'pipeline_tag': pipeline_tag,
    'run_id': run_id,
    'run_date_tag': run_date_tag,
    'target_column': target_column,
    'feature_columns': feature_columns,
    'risk_thresholds': risk_thresholds,
    'retention_rules': {
        'HIGH': {'base_discount_pct': 25, 'vip_discount_pct': 30, 'free_shipping': True, 'priority_level': 'immediate'},
        'MEDIUM': {'base_discount_pct': 12, 'vip_discount_pct': 12, 'free_shipping': True, 'priority_level': 'scheduled'},
        'LOW': {'base_discount_pct': 0, 'vip_discount_pct': 0, 'free_shipping': False, 'priority_level': 'light_touch'},
    },
    'dominant_driver_policy': dominant_driver_policy.to_dict(orient='records'),
    'test_snapshot_keys': package['test_snapshot_keys'],
    'source_artifacts': {
        'model': model_path.name,
        'feature': feature_path.name,
        'prediction': prediction_path.name,
        'explainability': explainability_path.name,
    },
}
deployment_bundle = {'metadata': inference_metadata, 'model_package': package}
joblib.dump(deployment_bundle, scoring_package_path)
logger.info('Scoring package saved to %s', scoring_package_path)
inference_metadata


2026-05-21 19:19:44,763 | INFO | Scoring package saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_scoring_package_20260506.joblib


{'model_name': 'xgboost',
 'version_name': 'v2',
 'model_version': 'v2_20260506',
 'pipeline_tag': 'canonical_v2c_phase2',
 'run_id': 'canonical_v2c_20260506',
 'run_date_tag': '20260506',
 'target_column': 'churn_v2_label',
 'feature_columns': ['total_orders',
  'delivered_orders',
  'total_items',
  'total_payment_value',
  'total_item_price',
  'total_freight_value',
  'avg_order_value',
  'avg_review_score',
  'max_installments',
  'distinct_payment_types',
  'distinct_products_total',
  'distinct_categories_total',
  'credit_card_value_total',
  'boleto_value_total',
  'voucher_value_total',
  'debit_card_value_total',
  'late_delivery_rate_total',
  'orders_30d',
  'delivered_orders_30d',
  'items_30d',
  'revenue_30d',
  'avg_order_value_30d',
  'avg_review_score_30d',
  'distinct_products_30d',
  'distinct_categories_30d',
  'credit_card_value_30d',
  'boleto_value_30d',
  'voucher_value_30d',
  'debit_card_value_30d',
  'late_delivery_rate_30d',
  'orders_60d',
  'delivered_or

In [4]:
scoring_module = dedent("""
import logging

import joblib
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)


def apply_risk_tier(probability: float, thresholds: dict) -> str:
    if probability >= thresholds['high_min_score']:
        return 'HIGH'
    if probability >= thresholds['medium_min_score']:
        return 'MEDIUM'
    return 'LOW'


def attach_retention_rules(scored: pd.DataFrame, metadata: dict) -> pd.DataFrame:
    retention_rules = metadata['retention_rules']
    scored = scored.copy()
    scored['recommended_discount_pct'] = scored['risk_tier'].map(lambda x: retention_rules[x]['base_discount_pct'])
    scored['free_shipping_flag'] = scored['risk_tier'].map(lambda x: retention_rules[x]['free_shipping'])
    scored['priority_level'] = scored['risk_tier'].map(lambda x: retention_rules[x]['priority_level'])
    vip_cutoff = scored['total_payment_value'].quantile(0.75) if 'total_payment_value' in scored.columns else np.inf
    scored['vip_human_touch_flag'] = (scored['risk_tier'].eq('HIGH') & (scored.get('total_payment_value', pd.Series(index=scored.index, dtype=float)).fillna(0) >= vip_cutoff))
    scored.loc[scored['vip_human_touch_flag'], 'recommended_discount_pct'] = retention_rules['HIGH']['vip_discount_pct']
    return scored


def score_dataframe(df: pd.DataFrame, package_path: str) -> pd.DataFrame:
    bundle = joblib.load(package_path)
    package = bundle['model_package']
    metadata = bundle['metadata']
    model = package['model']
    feature_columns = package['feature_columns']

    encoded = pd.get_dummies(df.copy(), columns=['customer_state'], dtype=float)
    encoded = encoded.reindex(columns=feature_columns, fill_value=0.0)
    probabilities = model.predict_proba(encoded)[:, 1]

    scored = df.copy()
    scored['churn_probability'] = probabilities
    scored['risk_tier'] = [apply_risk_tier(value, metadata['risk_thresholds']) for value in probabilities]
    scored = attach_retention_rules(scored, metadata)
    return scored
""")
scoring_module_path.write_text(scoring_module, encoding='utf-8')
logger.info('Scoring module written to %s', scoring_module_path)

2026-05-21 19:19:44,776 | INFO | Scoring module written to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/src/models/churn_scoring.py


In [5]:
leakage_columns = [
    'customer_unique_id', 'snapshot_key', 'snapshot_date', 'first_purchase_timestamp',
    'last_purchase_timestamp', 'future_orders_90d', 'future_revenue_90d', 'churn_90d_label',
    'churn_v2_label', 'next_purchase_timestamp', 'days_to_next_purchase', 'future_purchase_within_horizon'
]
latest_snapshot_key = sorted(feature_df['snapshot_key'].astype(str).unique())[-1]
latest_snapshot_df = feature_df[feature_df['snapshot_key'].astype(str) == latest_snapshot_key].copy()
model_input_df = latest_snapshot_df[[c for c in feature_df.columns if c not in leakage_columns]].copy()

namespace = {'dedent': dedent}
exec(scoring_module_path.read_text(encoding='utf-8'), namespace)
smoke_test_scored = namespace['score_dataframe'](model_input_df.head(500).copy(), str(scoring_package_path))
smoke_test_scored.insert(0, 'customer_unique_id', latest_snapshot_df.head(500)['customer_unique_id'].to_numpy())
smoke_test_scored.insert(1, 'snapshot_key', latest_snapshot_df.head(500)['snapshot_key'].to_numpy())
smoke_test_scored.insert(2, 'snapshot_date', latest_snapshot_df.head(500)['snapshot_date'].to_numpy())
smoke_test_scored.to_parquet(smoke_test_path, index=False)
logger.info('Inference smoke test saved to %s', smoke_test_path)
smoke_test_scored[['customer_unique_id', 'snapshot_key', 'churn_probability', 'risk_tier', 'recommended_discount_pct', 'free_shipping_flag', 'vip_human_touch_flag', 'priority_level']].head()

2026-05-21 19:19:44,818 | INFO | Inference smoke test saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/churn_inference_smoke_test_20260506.parquet


,customer_unique_id,snapshot_key,churn_probability,risk_tier,recommended_discount_pct,free_shipping_flag,vip_human_touch_flag,priority_level
7776,004288347e5e88a27ded2bb23747066c,20180501,0.944856,LOW,0,False,False,light_touch
7777,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180501,0.954483,LOW,0,False,False,light_touch
7778,011b4adcd54683b480c4d841250a987f,20180501,0.951397,LOW,0,False,False,light_touch
7779,013f4353d26bb05dc6652f1269458d8d,20180501,0.898583,LOW,0,False,False,light_touch
7780,015557c9912277312b9073947804a7ba,20180501,0.967295,MEDIUM,12,True,False,scheduled


---
## 7.3. NOTEBOOK CLOSURE


The deployment-preparation stage now has a reusable scoring package, a plain Python module for inference, and a smoke test proving that end-to-end prediction works outside the original training notebook.

The packaged logic is now aligned with the canonical V2C policy, including percentile-based risk tiers and core retention-rule outputs. The next notebook should translate these artifacts into an orchestration blueprint that n8n can run on a daily schedule with business-rule outputs already attached.